In [1]:

import pandas as pd

data = {
    'Sky': ['Sunny', 'Sunny', 'Rainy', 'Sunny'],
    'AirTemp': ['Warm', 'Warm', 'Cold', 'Warm'],
    'Humidity': ['Normal', 'High', 'High', 'High'],
    'Wind': ['Strong', 'Strong', 'Strong', 'Strong'],
    'Water': ['Warm', 'Warm', 'Warm', 'Cool'],
    'Forecast': ['Same', 'Same', 'Change', 'Change'],
    'EnjoySport': ['Yes', 'Yes', 'No', 'Yes']
}

df = pd.DataFrame(data)
df.to_csv('training_data.csv', index=False)

print("CSV file created successfully!")
print(df)


CSV file created successfully!
     Sky AirTemp Humidity    Wind Water Forecast EnjoySport
0  Sunny    Warm   Normal  Strong  Warm     Same        Yes
1  Sunny    Warm     High  Strong  Warm     Same        Yes
2  Rainy    Cold     High  Strong  Warm   Change         No
3  Sunny    Warm     High  Strong  Cool   Change        Yes


In [2]:

import pandas as pd
import csv

# Read training data from CSV file
data = pd.read_csv('training_data.csv')

attributes = list(data.columns[:-1])

# Initialize S and G boundaries
S = [['0'] * len(attributes)]
G = [['?'] * len(attributes)]

# Generalize S
def generalize(s, example):
    new_s = s.copy()
    for i in range(len(s)):
        if s[i] != example[i]:
            new_s[i] = '?'
    return new_s

# Check whether hypothesis covers an example
def covers(h, example):
    return all(h[i] == '?' or h[i] == example[i]
               for i in range(len(h)))

# Candidate-Elimination Algorithm
for _, row in data.iterrows():
    example = list(row[:-1])
    target = row.iloc[-1]

    if target == 'Yes':
        G = [g for g in G if covers(g, example)]

        new_S = []
        for s in S:
            if not covers(s, example):
                s = generalize(s, example)
            new_S.append(s)

        S = new_S

        G = [g for g in G if any(
            all(g[i] == '?' or g[i] == s[i]
                for i in range(len(g)))
            for s in S
        )]

    else:
        new_G = []

        for g in G:
            if covers(g, example):
                for i in range(len(g)):
                    if g[i] == '?':
                        for value in data[attributes[i]].unique():
                            if value != example[i]:
                                h = g.copy()
                                h[i] = value

                                if any(
                                    all(h[j] == '?' or h[j] == s[j]
                                        for j in range(len(h)))
                                    for s in S
                                ):
                                    new_G.append(h)
            else:
                new_G.append(g)

        G = new_G

    print("\nTraining Example:", example, target)
    print("S Boundary:", S)
    print("G Boundary:", G)

print("\nFinal Specific Boundary (S):")
for h in S:
    print(h)

print("\nFinal General Boundary (G):")
for h in G:
    print(h)



Training Example: ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same'] Yes
S Boundary: [['?', '?', '?', '?', '?', '?']]
G Boundary: [['?', '?', '?', '?', '?', '?']]

Training Example: ['Sunny', 'Warm', 'High', 'Strong', 'Warm', 'Same'] Yes
S Boundary: [['?', '?', '?', '?', '?', '?']]
G Boundary: [['?', '?', '?', '?', '?', '?']]

Training Example: ['Rainy', 'Cold', 'High', 'Strong', 'Warm', 'Change'] No
S Boundary: [['?', '?', '?', '?', '?', '?']]
G Boundary: []

Training Example: ['Sunny', 'Warm', 'High', 'Strong', 'Cool', 'Change'] Yes
S Boundary: [['?', '?', '?', '?', '?', '?']]
G Boundary: []

Final Specific Boundary (S):
['?', '?', '?', '?', '?', '?']

Final General Boundary (G):
